# Lesson 5.2 — PII Handling and Data Leakage Prevention

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Identify common PII categories in text
- Redact or tokenize PII before sending to the model
- Prevent sensitive data from leaking into logs and outputs
- Apply confidentiality and data-minimization principles

**You will produce:** A PII detector/redactor and a leak-safe logging wrapper.

## Concepts

**PII (personally identifiable information)** - emails, phone numbers, SSNs, credit cards, names - must be handled carefully for privacy and compliance. Principles:

- **Data minimization**: send only what the task needs; strip or tokenize PII first.
- **Redaction/tokenization**: replace PII with placeholders (reversible mapping if you must restore it) before prompts, logs, or traces.
- **Leak prevention**: never log raw prompts/outputs containing secrets or PII; scrub before writing. Watch for PII echoed back in model output.
- **Confidentiality & integrity**: enforce access controls on stored conversations and ensure data is not retained longer than needed.

> **Reference:** [Anthropic - Security & data handling](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Detect and redact PII

Use regex to find common PII and replace with typed placeholders.

In [ ]:
import re

PATTERNS = {
    "EMAIL": r"[\w.+-]+@[\w-]+\.[\w.-]+",
    "PHONE": r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b",
    "SSN": r"\b\d{3}-\d{2}-\d{4}\b",
    "CARD": r"\b(?:\d[ -]?){13,16}\b",
}

def redact(text):
    for label, pat in PATTERNS.items():
        text = re.sub(pat, f"[{label}]", text)
    return text

print(redact("Email dana@x.com or call 555-123-4567; SSN 123-45-6789"))

## Code-Along: Step 2 — Leak-safe logging

Scrub before logging so PII never lands in log files or traces.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("claude-app")

def safe_log(message, payload):
    logger.info("%s | %s", message, redact(payload))

safe_log("request", "user=dana@x.com asked about invoice 555-123-4567")

## Exercise

Build `tokenize(text)` that replaces each PII match with a stable token like `EMAIL_1` and returns `(clean_text, mapping)` so you can restore values after the model responds.

In [ ]:
def tokenize(text):
    # TODO: for each pattern, replace matches with LABEL_1, LABEL_2, ...
    # TODO: build a mapping {token: original}
    # TODO: return (clean_text, mapping)
    pass

## Key Takeaways

- Minimize data: send only what the task requires.
- Redact or tokenize PII before prompts, logs, and traces.
- Never log raw prompts/outputs with secrets or PII - scrub first.
- Watch for PII echoed back in model output and enforce access controls on stored data.

---

**Next:** [Lesson 5.3 — Guardrail Layering, Least Privilege, and Safety Hooks](./lesson-5.3-guardrails-and-least-privilege.ipynb)